# BGE-M3 임베딩 v4 — Colab A100

**입력**: `_chunks_output.jsonl` (v4 청커 출력, 3,135문서 / ~566K청크)

**출력**: `_embeddings_v3.jsonl` (chunk_uid, document_id, text, embedding)

## 준비
1. Google Drive 루트에 `_chunks_output.jsonl` 업로드
2. 런타임 → **A100 GPU** 선택
3. 셀 1부터 순서대로 실행

## Cell 1 — 환경 설치 + Drive 마운트

In [ ]:
!nvidia-smi | head -5

import subprocess
subprocess.run(['pip', 'install', '-q',
                'sentence-transformers==3.1.1',
                'torch', 'numpy'], check=True)

import torch
print(f'torch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')

import os, shutil, time

DRIVE_INPUT = '/content/drive/MyDrive/_chunks_output.jsonl'
LOCAL_INPUT = '/content/_chunks_output.jsonl'
OUTPUT_FILE = '/content/_embeddings_v3.jsonl'
DRIVE_OUTPUT = '/content/drive/MyDrive/_embeddings_v3.jsonl'
CHECKPOINT = '/content/_embed_checkpoint.json'

print('\n드라이브에서 입력 파일 복사...')
t0 = time.time()
shutil.copy(DRIVE_INPUT, LOCAL_INPUT)
size_mb = os.path.getsize(LOCAL_INPUT) / 1024 / 1024
print(f'완료! {size_mb:.0f} MB ({time.time()-t0:.1f}초)')

# 문서/청크 수 미리 카운트
import json
total_docs, total_chunks = 0, 0
with open(LOCAL_INPUT, 'r', encoding='utf-8') as f:
    for line in f:
        d = json.loads(line)
        total_docs += 1
        total_chunks += d.get('chunk_count', len(d.get('chunks', [])))

print(f'\n문서: {total_docs:,}건 | 청크: {total_chunks:,}개')
print('\n🟢 Cell 2 실행')

## Cell 2 — BGE-M3 모델 로드

In [ ]:
from sentence_transformers import SentenceTransformer
import torch, time

print('BGE-M3 모델 로드 (~2.2GB)...')
t0 = time.time()
model = SentenceTransformer('BAAI/bge-m3', device='cuda')
print(f'로드 완료! ({time.time()-t0:.1f}초)')

print(f'  pooling: {model[1].get_pooling_mode_str()}')
print(f'  max_seq: {model.max_seq_length}')
print(f'  dim:     {model.get_sentence_embedding_dimension()}')

# 워밍업
_ = model.encode(['워밍업'], normalize_embeddings=True)
print('워밍업 완료')

# A100 VRAM 기준 배치 크기 결정
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH_SIZE = 512 if vram_gb >= 70 else 256
print(f'\nVRAM {vram_gb:.0f}GB → 배치 크기: {BATCH_SIZE}')
print('\n🟢 Cell 3 실행')

## Cell 3 — 전체 임베딩 실행 (체크포인트 지원)

In [ ]:
import json, os, time
import numpy as np

SAVE_EVERY_CHUNKS = 10000   # 10K청크마다 Drive 자동 백업

# ── 체크포인트 복원 ──
completed_uids = set()
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                rec = json.loads(line)
                completed_uids.add(rec['chunk_uid'])
            except:
                pass
    print(f'체크포인트 복원: {len(completed_uids):,}건')
else:
    print('새로 시작')

# ── 플래트닝: (chunk_uid, document_id, text) 리스트 ──
pending = []
with open(LOCAL_INPUT, 'r', encoding='utf-8') as f:
    for line in f:
        doc = json.loads(line)
        rcept_no = doc.get('rcept_no', doc.get('filename', 'unk'))
        chunks = doc.get('chunks', [])
        for i, text in enumerate(chunks):
            uid = f'{rcept_no}_{i:05d}'
            if uid not in completed_uids:
                pending.append({
                    'chunk_uid': uid,
                    'document_id': rcept_no,
                    'company': doc.get('company', ''),
                    'category': doc.get('category', ''),
                    'text': text
                })

total_pending = len(pending)
total_all = total_chunks
print(f'\n총 청크: {total_all:,} | 완료: {len(completed_uids):,} | 남은 것: {total_pending:,}')

if total_pending == 0:
    print('✅ 이미 모두 완료!')
else:
    # ── 배치 임베딩 ──
    out_f = open(OUTPUT_FILE, 'a', encoding='utf-8')
    t_start = time.time()
    done = 0
    last_backup = 0

    for batch_start in range(0, total_pending, BATCH_SIZE):
        batch = pending[batch_start : batch_start + BATCH_SIZE]
        texts = [r['text'] for r in batch]

        embs = model.encode(
            texts,
            normalize_embeddings=True,
            batch_size=BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
        )

        for rec, emb in zip(batch, embs):
            out_rec = {
                'chunk_uid': rec['chunk_uid'],
                'document_id': rec['document_id'],
                'company': rec['company'],
                'category': rec['category'],
                'text': rec['text'],
                'embedding': emb.tolist(),
            }
            out_f.write(json.dumps(out_rec, ensure_ascii=False) + '\n')

        done += len(batch)

        # 진행 로그
        if done % 5000 < BATCH_SIZE or done == total_pending:
            elapsed = time.time() - t_start
            rate = done / elapsed if elapsed > 0 else 0
            eta_min = (total_pending - done) / rate / 60 if rate > 0 else 0
            pct = (len(completed_uids) + done) / total_all * 100
            print(f'  [{done:,}/{total_pending:,}] {pct:.1f}% | {rate:.0f}청크/초 | ETA {eta_min:.1f}분')

        # 자동 Drive 백업
        if done - last_backup >= SAVE_EVERY_CHUNKS:
            out_f.flush()
            import shutil
            shutil.copy(OUTPUT_FILE, DRIVE_OUTPUT)
            last_backup = done
            print(f'  💾 Drive 백업 ({done:,}청크)')

    out_f.close()

    elapsed = time.time() - t_start
    size_mb = os.path.getsize(OUTPUT_FILE) / 1024 / 1024
    print(f'\n{'='*55}')
    print(f'  ✅ 임베딩 완료!')
    print(f'  총 청크: {total_all:,} | 소요: {elapsed/60:.1f}분')
    print(f'  출력 파일: {size_mb:.0f} MB')
    print(f'{'='*55}')

    # 최종 Drive 저장
    import shutil
    shutil.copy(OUTPUT_FILE, DRIVE_OUTPUT)
    print('💾 Drive 최종 저장 완료')
    print('\n🟢 Cell 4: 다운로드 또는 Drive에서 직접 가져가세요')

## Cell 4 — 검증 + 다운로드

In [ ]:
import json, os
import numpy as np

# 빠른 검증 (첫 5개 + 랜덤 10개)
records = []
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 5:
            records.append(json.loads(line))

total_lines = i + 1
print(f'총 레코드: {total_lines:,}')

r = records[0]
emb = np.array(r['embedding'])
print(f'\n샘플 chunk_uid: {r["chunk_uid"]}')
print(f'embedding dim: {emb.shape[0]}')
print(f'embedding norm: {np.linalg.norm(emb):.4f} (정상=1.0)')
print(f'text 앞 100자: {r["text"][:100]}')

file_mb = os.path.getsize(OUTPUT_FILE) / 1024 / 1024
print(f'\n파일 크기: {file_mb:.0f} MB')

if emb.shape[0] == 1024 and abs(np.linalg.norm(emb) - 1.0) < 0.01:
    print('\n✅ 검증 통과 — dim=1024, norm≈1.0')
else:
    print('\n❌ 검증 실패!')

# 다운로드 (파일이 작으면 직접, 크면 Drive에서 가져가세요)
if file_mb < 2000:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print('다운로드 시작!')
else:
    print(f'\n파일이 {file_mb:.0f}MB로 큽니다.')
    print('Google Drive에서 직접 다운로드 → 로컬 Omega_CivicFlow_v4_DB/_embeddings_v3.jsonl 으로 저장')